<a href="https://colab.research.google.com/github/ankit-rathi/Quantvesting_v2/blob/main/myStocks_Portfolio_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
'''
!pip install pandas==2.2.2 -qq
!pip install pyxirr -qq
!pip install ta -qq
!pip install yfinance==0.2.59 -qq
'''

'\n!pip install pandas==2.2.2 -qq\n!pip install pyxirr -qq\n!pip install ta -qq\n!pip install yfinance==0.2.59 -qq\n'

In [2]:
# import libraries

import numpy as np
import pandas as pd
import datetime
import warnings
warnings.filterwarnings('ignore')
import yfinance as yfin
import ta
import matplotlib.pyplot as plt
import requests

In [3]:
# notebook start time
import time
import datetime
import pytz

start_time = time.time()

# function to print date timestamp
def print_date_time():
  tz_NY = pytz.timezone('Asia/Kolkata')
  datetime_NY = datetime.datetime.now(tz_NY)
  print("Run date time (IST):", datetime_NY.strftime("%Y-%m-%d %H:%M:%S"))

In [4]:
# function to get booked and reserved amount
def get_amt():
  # fetch the JSON data from the URL
  url = "https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioAmts.json"
  response = requests.get(url)
  pf_amts = response.json()  # parse the JSON data

  # extract values from the JSON
  py_booked_amt_dm = pf_amts["py_booked_amt_dm"]
  py_booked_amt_sv = pf_amts["py_booked_amt_sv"]
  cy_booked_amt_dm = pf_amts["cy_booked_amt_dm"]
  cy_booked_amt_sv = pf_amts["cy_booked_amt_sv"]
  reserve_amt_dm = pf_amts["reserve_amt_dm"]
  reserve_amt_sv = pf_amts["reserve_amt_sv"]

  # perform calculations
  py_booked_amt = py_booked_amt_dm + py_booked_amt_sv
  cy_booked_amt = cy_booked_amt_dm + cy_booked_amt_sv
  reserve_amt = reserve_amt_dm + reserve_amt_sv
  total_booked_amt = py_booked_amt + cy_booked_amt

  return total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt

gen_output = 0

In [5]:
# function to format the amount
def format_amt(number):
    abs_number = abs(number)

    if abs_number >= 1_00_00_000:  # Crores
        formatted_amt = f"{number / 1_00_00_000:.2f} C"
    elif abs_number >= 1_00_000:  # Lakhs
        formatted_amt = f"{number / 1_00_000:.2f} L"
    elif abs_number >= 1_000:  # Thousands
        formatted_amt = f"{number / 1_000:.2f} K"
    else:
        formatted_amt = f"{number:.2f}"

    return formatted_amt

# function to set start and end date
def get_start_end_date():
  start_date = (datetime.date.today() + datetime.timedelta(days=-365)).strftime('%Y-%m-%d')
  end_date = (datetime.date.today() + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
  return start_date, end_date

# function to fetch my portfolio csv
def get_mypfs_df():
  mypfs_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myPortfolioStocks.csv')
  return mypfs_df

# function to fetch my prospects csv
def get_mypps_df():
  mypps_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myProspectsScrips.csv')
  return mypps_df

# function to fetch screener data
def get_myscreen_df():
  myscreen_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myScreenerDB.csv')
  return myscreen_df

# function to fetch momentum data
def get_myinvmt_df():
  myinvmt_df = pd.read_csv('https://raw.githubusercontent.com/ankit-rathi/Tradevesting_v1/main/data/myInvestments.csv')
  return myinvmt_df

# function to get the stock ids
def get_stock_ids(df_pf):
  stock_n100 = df_pf['Symbol'].unique()

  exclude = ['CADILAHC','MMTC', 'MASFIN']

  stock_ids = df_pf[~df_pf['Symbol'].isin(exclude) ]['Symbol'].unique()

  #mypf = mypf[mypf.Forecast.notnull()]
  #stock_ids = mypf['Symbol'].unique()

  #stock_ids.sort()
  return stock_ids

# get features from screener data
def get_screener_features():
  myscreen_df = get_myscreen_df()
  cols = ['Symbol', 'EPS', 'MedPE', 'ROCE%', 'ROE%', 'CapType']
  return myscreen_df[cols]

# get relative strength
def get_relative_strength(stock_list):

    # Dictionary to store stock tickers and their corresponding percentage price change
    stock_changes = {}

    # Loop through each stock and fetch its price data
    for stock in stock_list:
        try:
            # Download the stock data for the given date range
            stock_data = yfin.Ticker(stock + '.NS').history(period='1mo', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

            # Calculate the percentage change for the stock
            if len(stock_data) > 0:
                start_price = stock_data['Close'].iloc[0]
                end_price = stock_data['Close'].iloc[-1]
                percent_change = round(((end_price - start_price) / start_price) * 100, 2)
                stock_changes[stock] = percent_change
            else:
                stock_changes[stock] = np.nan  # If no data is available, set to NaN

        except Exception as e:
            print(f"Error fetching data for {stock}: {e}")
            stock_changes[stock] = np.nan

    # Create a DataFrame with stock tickers and their percentage changes
    df = pd.DataFrame(list(stock_changes.items()), columns=['Symbol', 'Percent_Change'])

    # Drop any stocks with missing data (NaN values)
    df = df.dropna()

    # Calculate the percentile rank based on percentage change
    df['RSP'] = round(df['Percent_Change'].rank(pct=True) * 100, 2)
    cols = ['Symbol', 'RSP']
    df = df[cols]
    # Sort by percentile rank (optional)
    df = df.sort_values(by='RSP', ascending=False).reset_index(drop=True)

    return df


In [6]:
# function to get stock technicals
def stock_prec_dev(stock_symbol):
    #stock_symbol = 'ULTRACEMCO.NS'
    short_window = 20
    mid_window = 50
    long_window = 200
    moving_avg = 'SMA'

    start = datetime.datetime(*map(int, start_date.split('-')))
    end = datetime.datetime(*map(int, end_date.split('-')))

    stock_df = yfin.Ticker(stock_symbol + '.NS').history(period='max', interval='1d')[map(str.title, ['open', 'close', 'low', 'high', 'volume'])]

    stock_df['Max'] = round(max(stock_df['Close']),2)
    stock_df = stock_df[(stock_df.index <= end_date) & (stock_df.index >= start_date)]
    stock_df['200_SMA'] = round(stock_df['Close'].rolling(window = 200, min_periods = 1).mean(),0)
    stock_df['Dev%_200'] = round((stock_df['Close'] - stock_df['200_SMA'])*100/stock_df['200_SMA'],2)
    stock_df.dropna(axis = 0, inplace = True) # remove any null rows

    stock_df['20_SMA'] = stock_df['Close'].rolling(window=20).mean()
    stock_df['50_SMA'] = stock_df['Close'].rolling(window=50).mean()
    stock_df['Symbol'] = stock_symbol

    stock_df['Close'] = round(stock_df['Close'],2)
    stock_df['Min'] = round(min(stock_df['Close']),2)
    stock_df['RSI_14'] = ta.momentum.RSIIndicator(close=stock_df['Close'], window=14).rsi()
    stock_df['RSI_14'] = round(stock_df['RSI_14'],0)
    stock_df['Prev_Close'] = stock_df['Close'].shift(1)
    stock_df.drop(['Open', 'Low', 'High', 'Volume'], axis=1, inplace=True)
    stock_df = stock_df.tail(1)

    max_SMA = max(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    min_SMA = min(stock_df['20_SMA'].item(), stock_df['50_SMA'].item(), stock_df['200_SMA'].item())
    ABS_Spread = max_SMA - min_SMA
    stock_df['Spread%'] = round((ABS_Spread / stock_df['200_SMA'].item()) * 100,2)

    return stock_df

# function to compute stock attributes
def get_common_features(stock_ids, df_mypf):

  df_prec_dev = pd.DataFrame()
  df_tmp = get_screener_features()
  df_rs = get_relative_strength(stock_ids)

  for stock_id in stock_ids:
      tmp = stock_prec_dev(stock_id)
      tmp = tmp.reset_index()
      df_prec_dev = pd.concat([df_prec_dev, tmp], ignore_index = True)
  df_prec_dev = pd.merge(df_prec_dev, df_mypf, on= 'Symbol')
  df_prec_dev = pd.merge(df_prec_dev, df_tmp, on= 'Symbol', how='left')
  df_prec_dev = pd.merge(df_prec_dev, df_rs, on= 'Symbol', how='left')
  #print(df_prec_dev.columns)
  df_prec_dev['Curr_PE'] = round(df_prec_dev['Close']/df_prec_dev['EPS'],1)
  df_prec_dev['Dev%_PE'] = round((df_prec_dev['Curr_PE'] - df_prec_dev['MedPE'])*100/df_prec_dev['MedPE'],2)
  df_prec_dev['Conviction'] = df_prec_dev['Conviction'] + '-' + df_prec_dev['CapType']
  return df_prec_dev

# function to arrange stock features
def arrange_features(df_stocks, common_cols, diff_cols):
  df_stocks_common = df_stocks[common_cols].drop_duplicates()
  df_stocks_diff = df_stocks[diff_cols]
  df_stocks_diff['Investment'] = df_stocks_diff['AvgCost'] * df_stocks_diff['Shares']
  df_stocks_diff = df_stocks_diff.groupby(['Symbol'])[['Shares', 'Investment']].aggregate(['sum']).reset_index()
  df_stocks_diff.columns = ['Symbol', 'Shares', 'Investment']
  df_stocks_diff['AvgCost'] = round(df_stocks_diff['Investment']/df_stocks_diff['Shares'],2)
  df_stocks = pd.merge(df_stocks_diff, df_stocks_common, on='Symbol')
  return df_stocks

# plot fact distribution across dimension
def plot_pie_chart(df, dimension, fact):
  # grouping the data by category and calculating the sum of fact for each type
  grouped_data = df.groupby(dimension)[fact].sum()

  # sorting the grouped data in descending order
  grouped_data = grouped_data.sort_values(ascending=False)

  # creating a pie chart
  grouped_data.plot.pie(autopct='%1.1f%%', startangle=90, figsize=(6, 6))

  # adding a title and displaying the plot
  plt.title(f'{dimension} {fact} Distribution')
  plt.ylabel('')  # To hide the y-label
  plt.show()

In [7]:
# function to get portfolio features

def get_portfolio_features(df_common_features):

  print_date_time()
  print('=================== PORTFOLIO QUALITY ====================')

  #df_common_features["Target"] = df_common_features["Target"].fillna(df_common_features["Max"])
  df_common_features['NTT'] = np.where(df_common_features['Strategy']== 'NTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['LTT'] = np.where(df_common_features['Strategy']== 'BTT', df_common_features["Target"], df_common_features['Max'])
  df_common_features['BOL'] = df_common_features['Min']

  tmp_df = df_common_features[df_common_features['Symbol'].isin(mypf_df[mypf_df['InPortfolio'] != 'NA'].Symbol.values)]
  print('qualified stocks: '+str(len(tmp_df['Symbol'].unique())))
  tmp_df1 = tmp_df[tmp_df['LatestQtr'] == 1]
  print('with latest results: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df1 = tmp_df1[tmp_df1['StarStock'] == 1]
  print('still star stocks: '+str(len(tmp_df1['Symbol'].unique())))
  tmp_df['Investment'] = tmp_df['AvgCost'] * tmp_df['Shares']
  tmp_df['Current'] = round(tmp_df['Close'] * tmp_df['Shares'],0)
  tmp_df['Previous'] = tmp_df['Prev_Close'] * tmp_df['Shares']
  tmp_df['EstimatedST'] = tmp_df['NTT'] * tmp_df['Shares']
  tmp_df['EstimatedLT'] = tmp_df['LTT'] * tmp_df['Shares']
  tmp_df['Current P/L'] = round((tmp_df['Current'] - tmp_df['Investment']),0)
  tmp_df['Today P/L%'] = round((tmp_df['Current'] - tmp_df['Previous'])*100/tmp_df['Previous'],2)
  tmp_df['Current P/L%'] = round((tmp_df['Current'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedST P/L%'] = round((tmp_df['EstimatedST'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['EstimatedLT P/L%'] = round((tmp_df['EstimatedLT'] - tmp_df['Investment'])*100/tmp_df['Investment'],2)
  tmp_df['NTT%'] = round((tmp_df['NTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['LTT%'] = round((tmp_df['LTT'] - tmp_df['Close'])*100/tmp_df['Close'],2)
  tmp_df['Gained%'] = round((tmp_df['Close'] - tmp_df['BOL'])*100/tmp_df['BOL'],2)
  investment = round(sum(tmp_df['AvgCost']*tmp_df['Shares']),0)
  current = round(sum(tmp_df['Close']*tmp_df['Shares']),0)
  tmp_df['InitAlloc%'] = round(tmp_df['Investment']*100/investment,2)
  tmp_df['CurrAlloc%'] = round(tmp_df['Current']*100/current,2)
  tmp_df['FTT'] = tmp_df['LTT']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT'] = tmp_df['NTT']
  tmp_df['FTT%'] = tmp_df['LTT%']
  tmp_df.loc[tmp_df['Strategy'] == 'NTT', 'FTT%'] = tmp_df['NTT%']
  tmp_df['FTT Amt'] = round(tmp_df['FTT%'] * tmp_df['Current']/100,0)
  tmp_df['OTT%'] = round((tmp_df['FTT'] - tmp_df['AvgCost'])*100/tmp_df['AvgCost'],2)
  tmp_df['RRR Ind'] = round(tmp_df['Current P/L']/tmp_df['FTT Amt'],2)
  tmp_df['Risk Ind'] = round(tmp_df['Current P/L%']*tmp_df['CurrAlloc%'],0)

  return tmp_df

# function to print portfolio features
def print_portfolio_stats(df_portfolio_features, myinvmt_df):
  from pyxirr import xirr

  total_booked_amt, reserve_amt, cy_booked_amt, py_booked_amt = get_amt()

  dates = myinvmt_df['Date'].values
  dates = np.append(dates, datetime.date.today().strftime('%d-%b-%y'))
  investment = myinvmt_df['Investment'].values
  dates= pd.to_datetime(dates)

  current = round(sum(df_portfolio_features['Close']*df_portfolio_features['Shares']),0) + reserve_amt
  investment_xirr = np.append(investment, current)
  cagr = round(xirr(pd.DataFrame({"dates": dates, "amounts": investment_xirr}))*100,2)

  investment = -sum(investment)
  invested = round(sum(df_portfolio_features['AvgCost']*df_portfolio_features['Shares']),0) + reserve_amt
  previous = round(sum(df_portfolio_features['Prev_Close']*df_portfolio_features['Shares']),0) + reserve_amt
  cy_invested = investment + py_booked_amt

  today_pnl_amount = current-previous
  today_pnl_percentage = round((current-previous)*100/previous,2)

  curr_pnl_amount = current-invested
  curr_pnl_percentage = round((curr_pnl_amount)*100/(cy_invested),2)

  cy_pnl_amount = cy_booked_amt + curr_pnl_amount
  cy_pnl_percentage = round((cy_pnl_amount)*100/cy_invested,2)

  overall_pnl_amount = total_booked_amt + curr_pnl_amount
  overall_pnl_percentage = round((overall_pnl_amount)*100/investment,2)

  estimate_st = round(sum(df_portfolio_features['FTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_st_pnl_amount = estimate_st-current
  est_st_pnl_percentage = round((est_st_pnl_amount)*100/current,2)

  estimate_lt = round(sum(df_portfolio_features['LTT']*df_portfolio_features['Shares']),0)  + reserve_amt
  est_lt_pnl_amount = estimate_lt-current
  est_lt_pnl_percentage = round((est_lt_pnl_amount)*100/current,2)

  total_profit = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] > 0]['Investment']),0)
  total_loss = round(sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Current']) - sum(df_portfolio_features[df_portfolio_features['Current P/L%'] < 0]['Investment']),0)

  cy_booked_percentage = round((cy_booked_amt)*100/current,2)
  py_booked_percentage = round((py_booked_amt)*100/investment,2)
  total_booked_percentage = round((total_booked_amt)*100/investment,2)

  print('=================== PORTFOLIO SNAPSHOT ===================')
  print('Deployed Value: ', format_amt(investment))
  print('Current Value:  ', format_amt(current))
  print('CAGR/XIRR %:    ', '{}%'.format(cagr))
  print('Est FTT Value:  ', format_amt(estimate_st))
  print('Future Upside:  ', '{} ({}%)'.format(
      format_amt(est_st_pnl_amount),
      est_st_pnl_percentage
  ))

  print('=================== PERFORMANCE ==========================')
  print('Current PnL:             {} ({}%)'.format(
      format_amt(curr_pnl_amount), curr_pnl_percentage
  ))
  print('Total Booked PnL:        {} ({}%)'.format(
      format_amt(total_booked_amt), total_booked_percentage
  ))
  print('Booked + Current PnL:    {} ({}%)'.format(
      format_amt(overall_pnl_amount), overall_pnl_percentage
  ))
  print('CY Booked PnL:           {} ({}%)'.format(
      format_amt(cy_booked_amt), cy_booked_percentage
  ))
  print('Prev Year Booked PnL:    {} ({}%)'.format(
      format_amt(py_booked_amt), py_booked_percentage
  ))

  print('=================== TODAY / OPERATIONS ===================')
  print('Today PnL:               {} ({}%)'.format(
      format_amt(today_pnl_amount), today_pnl_percentage
  ))
  print('CY Investment:           ', format_amt(cy_invested))
  print('Reserve:                 ', format_amt(reserve_amt))

  print('=================== P&L DETAIL ============================')
  print('Total Profit:            ', format_amt(total_profit))
  print('Total Loss:              ', format_amt(total_loss))
  print('============================================================')

In [8]:
# get start and end date
start_date, end_date = get_start_end_date()

# get portfolio and prospects data
mypfs_df = get_mypfs_df()
mypps_df = get_mypps_df()
myinvmt_df = get_myinvmt_df()

# merge above datasets
mypf_df = pd.merge(mypfs_df, mypps_df, on="Symbol")

# seggregate dm and sv portfolio
dm_pf = mypf_df[mypf_df['InPortfolio'] == 'DM']
sv_pf = mypf_df[mypf_df['InPortfolio'] == 'SV']

# build portfolio stock dataframe
dm_stocks = get_stock_ids(dm_pf)
sv_stocks = get_stock_ids(sv_pf)
df_stocks = pd.concat([dm_pf,sv_pf], ignore_index = True)

# arrange common and diff stock features
common_cols = ['Symbol', 'Target', 'Criteria', 'Strategy', 'CumlRnk', 'LatestQtr', 'StarStock', 'Conviction', 'Category', 'InFolio', 'XIRR', 'MBQ']
diff_cols = ['Symbol', 'AvgCost', 'Shares']
df_stocks = arrange_features(df_stocks, common_cols, diff_cols)

# get common features
stock_ids = df_stocks['Symbol'].values
df_common_features = get_common_features(stock_ids, df_stocks)

df_common_features.reset_index(drop=True, inplace=True)
df_common_features.drop(['Date'], axis=1, inplace=True)
# get and print portfolio features
df_portfolio_features = get_portfolio_features(df_common_features)

print_portfolio_stats(df_portfolio_features, myinvmt_df)

df = df_portfolio_features
#plot_pie_chart(df, 'CapType', 'Current')
list_ox40n = list(df[df['MBQ'].str.contains('OX40N', na=False)]['Symbol'].values)

Run date time (IST): 2026-11-19 15:24:24
=================== PORTFOLIO QUALITY ====================
qualified stocks: 84
with latest results: 84
still star stocks: 44
=================== PORTFOLIO SNAPSHOT ===================
Deployed Value:  1.49 C
Current Value:   1.54 C
CAGR/XIRR %:     1.33%
Est FTT Value:   2.74 C
Future Upside:   1.20 C (77.76%)
=================== PERFORMANCE ==========================
Current PnL:             -41.30 L (-21.52%)
Total Booked PnL:        49.23 L (33.02%)
Booked + Current PnL:    7.93 L (5.32%)
CY Booked PnL:           6.36 L (4.13%)
Prev Year Booked PnL:    42.87 L (28.76%)
=================== TODAY / OPERATIONS ===================
Today PnL:               2.01 L (1.33%)
CY Investment:            1.92 C
Reserve:                  2.04 K
=================== P&L DETAIL ============================
Total Profit:             2.76 L
Total Loss:               -44.06 L


In [9]:
# top 5 near their targets
cols = ['Symbol', 'Today P/L%', 'Current P/L%', 'FTT%', 'OTT%', 'FTT Amt', 'Current P/L', 'Current', 'FTT', 'Dev%_PE', 'RSI_14','Conviction', 'Spread%',  'CumlRnk', 'RRR Ind', 'CurrAlloc%', 'Gained%', 'Criteria', 'Strategy', 'Category']
df_tmp = df_portfolio_features[~df_portfolio_features['Symbol'].isin(['ENRIN','BLUSPRING','DIGITIDE'])]
df_tmp = df_tmp.sort_values(by = 'FTT Amt', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
50,M&M,4.66,9.76,7.04,17.48,23264.0,29382.0,330457.0,3762.88,-2.50,73.0,X-LC,4.37,25.0,1.26,2.15,27.78,X40N,ATH,AUTO
75,TTKPRESTIG,0.97,-27.50,38.05,0.09,27801.0,-27712.0,73065.0,770.00,-26.64,53.0,M-SC,2.62,250.0,-1.00,0.47,31.51,OX40N,NTT,DURABLES
12,BOSCH-HCIL,1.23,-22.88,29.66,-0.01,29233.0,-29245.0,98560.0,2282.00,772.08,52.0,M-SC,18.82,231.0,-1.00,0.64,67.76,OX40N,BTT,AC
76,UNITDSPR,1.67,7.62,12.22,20.77,30792.0,17836.0,251980.0,1644.00,-5.70,63.0,X-MC,0.68,64.0,0.58,1.64,21.13,X40N,NTT,BREWERIES
52,MASFIN,-0.54,-19.88,51.35,21.26,40309.0,-19482.0,78498.0,396.02,-38.40,48.0,H-SC,14.50,167.0,-0.48,0.51,6.95,XR,ATH,FINANCE


In [10]:
# top 5 to accumulate
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 40) & (df_tmp['FTT%'] > 20)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
44,INFY,-1.64,-21.10,80.82,42.66,255613.0,-84581.0,316274.0,1972.00,-42.51,44.0,X-LC,6.62,3.0,-0.33,2.05,13.22,X40,NTT,IT
47,JIOFIN,0.98,-20.28,70.71,36.09,226343.0,-81445.0,320100.0,387.00,-29.25,56.0,X-LC,7.96,34.0,-0.36,2.08,10.86,XY24,BTT,FINANCE
38,HYUNDAI,6.08,13.77,25.07,42.29,80882.0,39054.0,322625.0,2782.73,32.77,67.0,X-LC,2.18,29.0,0.48,2.10,32.80,X40N,ATH,AUTO
33,HCLTECH,-1.32,-9.20,38.44,25.70,125994.0,-33204.0,327769.0,1837.07,-15.16,41.0,X-LC,9.34,8.0,-0.26,2.13,30.77,X40,ATH,IT
4,ASIANPAINT,0.49,-5.43,31.99,24.83,107831.0,-19340.0,337077.0,3422.40,-16.67,65.0,X-LC,2.97,30.0,-0.18,2.19,23.58,X40,ATH,PAINTS


In [11]:
# top 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
81,VOLTAS,8.59,-3.61,50.64,45.20,93590.0,-6927.0,184815.0,1856.00,5.16,72.0,X-MC,16.19,79.0,-0.07,1.20,30.22,XY25,NTT,AC
51,MARUTI,7.30,-0.25,26.31,25.99,89151.0,-860.0,338850.0,17120.06,3.02,72.0,X-LC,8.87,33.0,-0.01,2.20,21.34,XR,ATH,AUTO
38,HYUNDAI,6.08,13.77,25.07,42.29,80882.0,39054.0,322625.0,2782.73,32.77,67.0,X-LC,2.18,29.0,0.48,2.10,32.80,X40N,ATH,AUTO
32,HAVELLS,5.17,-23.24,71.21,31.42,179285.0,-76222.0,251769.0,2052.62,-34.34,66.0,X-MC,10.11,63.0,-0.43,1.64,18.88,X40,ATH,ELECTRICAL
50,M&M,4.66,9.76,7.04,17.48,23264.0,29382.0,330457.0,3762.88,-2.50,73.0,X-LC,4.37,25.0,1.26,2.15,27.78,X40N,ATH,AUTO


In [12]:
# bottom 5 today
df_tmp = df_portfolio_features.sort_values(by = 'Today P/L%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-3.83,-24.13,183.62,115.17,377189.0,-65343.0,205418.0,1903.93,-17.22,59.0,H-SC,24.54,147.0,-0.17,1.33,81.59,AR,ATH,IT
78,VAIBHAVGBL,-2.52,-50.64,225.73,60.77,203636.0,-92563.0,90212.0,521.00,NaN,32.0,NaN,22.05,160.0,-0.45,0.59,3.99,XR,NTT,JEWELLERY
24,EASEMYTRIP,-2.03,-65.45,321.73,45.70,196529.0,-115730.0,61085.0,26.40,NaN,61.0,NaN,16.30,204.0,-0.59,0.40,17.45,XY24,NTT,TRAVEL
44,INFY,-1.64,-21.10,80.82,42.66,255613.0,-84581.0,316274.0,1972.00,-42.51,44.0,X-LC,6.62,3.0,-0.33,2.05,13.22,X40,NTT,IT
71,TCS,-1.34,-29.30,79.27,26.75,273547.0,-142993.0,345082.0,4208.34,-43.36,43.0,X-LC,7.29,2.0,-0.52,2.24,19.69,X40,ATH,IT


In [13]:
# top 5 to exit based on CumlRnk
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > -1) & (df_tmp['Current P/L%'] < 1)].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category


In [14]:
# OX40N stocks to exit
df_tmp = df_portfolio_features[(df_portfolio_features['Symbol'].isin(list_ox40n))].sort_values(by = 'Current P/L%', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.64,52.68,35.21,106.44,79373.0,77784.0,225428.0,1629.94,-4.52,47.0,M-SC,22.41,220.0,0.98,1.46,70.81,OX40N,ATH,CABLES
65,SIS,-0.87,-16.51,45.23,21.26,41930.0,-18327.0,92705.0,528.00,78.35,35.0,H-SC,5.46,168.0,-0.44,0.60,37.84,OX40N,NTT,MISC
12,BOSCH-HCIL,1.23,-22.88,29.66,-0.01,29233.0,-29245.0,98560.0,2282.00,772.08,52.0,M-SC,18.82,231.0,-1.00,0.64,67.76,OX40N,BTT,AC
75,TTKPRESTIG,0.97,-27.50,38.05,0.09,27801.0,-27712.0,73065.0,770.00,-26.64,53.0,M-SC,2.62,250.0,-1.00,0.47,31.51,OX40N,NTT,DURABLES
42,INDIGOPNTS,1.07,-28.87,40.62,0.02,50432.0,-50404.0,124155.0,1408.00,-32.23,53.0,M-SC,6.95,222.0,-1.00,0.81,41.46,OX40N,NTT,PAINTS


In [15]:
# top 5 to book for rotation
df_tmp = df_portfolio_features[(df_portfolio_features['Current P/L%'] > 20) ].sort_values(by = 'CumlRnk', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.64,52.68,35.21,106.44,79373.0,77784.0,225428.0,1629.94,-4.52,47.0,M-SC,22.41,220.0,0.98,1.46,70.81,OX40N,ATH,CABLES


In [16]:
# top 5 to monitor for exit
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['Current P/L%'] > 0) ].sort_values(by = 'Dev%_PE', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.64,52.68,35.21,106.44,79373.0,77784.0,225428.0,1629.94,-4.52,47.0,M-SC,22.41,220.0,0.98,1.46,70.81,OX40N,ATH,CABLES
66,STARHEALTH,1.77,10.24,33.58,47.26,104070.0,28794.0,309917.0,761.00,-19.51,57.0,H-SC,3.61,173.0,0.28,2.01,34.00,XY24,NTT,INSURANCE


In [17]:
# top 5 to book for rotation from weak conviction
df_tmp = df_portfolio_features[~df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[~df_tmp['Criteria'].isin(['XY25','XY24','X40', 'X40N'])].sort_values(by = 'RRR Ind', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
25,FINCABLES,1.64,52.68,35.21,106.44,79373.0,77784.0,225428.0,1629.94,-4.52,47.0,M-SC,22.41,220.0,0.98,1.46,70.81,OX40N,ATH,CABLES
83,WIPRO,-0.32,-15.19,127.59,93.01,270773.0,-38019.0,212221.0,420.00,-27.04,45.0,M-LC,8.22,98.0,-0.14,1.38,17.70,XR,NTT,IT
69,TANLA,-3.83,-24.13,183.62,115.17,377189.0,-65343.0,205418.0,1903.93,-17.22,59.0,H-SC,24.54,147.0,-0.17,1.33,81.59,AR,ATH,IT
41,INDIAMART,0.24,-33.27,201.09,100.92,165505.0,-41032.0,82304.0,4675.60,-60.27,46.0,H-SC,16.58,140.0,-0.25,0.53,4.42,AR,ATH,MISC
3,ALKYLAMINE,-0.71,-32.96,187.22,92.54,126714.0,-33281.0,67682.0,4520.85,-43.63,37.0,H-SC,8.06,142.0,-0.26,0.44,28.52,SR,ATH,CHEMICALS


In [18]:
# top 5 to accumulate based on lowest RSI_14
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'RSI_14', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
54,PAGEIND,0.75,-19.04,63.31,32.21,137560.0,-51099.0,217280.0,50690.44,-38.63,31.0,X-SC,6.16,55.0,-0.37,1.41,4.26,X40,ATH,APPARELS
37,HONAUT,0.74,-22.07,93.62,50.88,140687.0,-42563.0,150275.0,58192.80,-38.96,40.0,X-SC,8.97,90.0,-0.30,0.98,14.49,X40N,ATH,ELECTRICAL
73,TMPV,3.76,-34.66,128.96,49.60,401810.0,-165295.0,311577.0,600.00,-531.72,40.0,X-MC,17.18,4.0,-0.41,2.02,3.76,XY24,NTT,AUTO
33,HCLTECH,-1.32,-9.20,38.44,25.70,125994.0,-33204.0,327769.0,1837.07,-15.16,41.0,X-LC,9.34,8.0,-0.26,2.13,30.77,X40,ATH,IT
71,TCS,-1.34,-29.30,79.27,26.75,273547.0,-142993.0,345082.0,4208.34,-43.36,43.0,X-LC,7.29,2.0,-0.52,2.24,19.69,X40,ATH,IT


In [19]:
# top 5 to accumulate based on Spread and CumlRnk
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])]
df_tmp = df_tmp[(df_tmp['CumlRnk'] < 100)].sort_values(by = 'Spread%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
76,UNITDSPR,1.67,7.62,12.22,20.77,30792.0,17836.0,251980.0,1644.00,-5.70,63.0,X-MC,0.68,64.0,0.58,1.64,21.13,X40N,NTT,BREWERIES
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA
17,CIPLA,-0.05,6.46,16.37,23.89,52398.0,19429.0,320084.0,1648.14,39.87,62.0,X-MC,1.90,18.0,0.37,2.08,19.89,X40N,ATH,PHARMA
38,HYUNDAI,6.08,13.77,25.07,42.29,80882.0,39054.0,322625.0,2782.73,32.77,67.0,X-LC,2.18,29.0,0.48,2.10,32.80,X40N,ATH,AUTO
11,BERGEPAINT,-0.20,-14.09,45.86,25.32,101555.0,-36304.0,221445.0,680.00,-23.24,46.0,X-MC,2.53,68.0,-0.36,1.44,16.12,XY24,NTT,PAINTS


In [20]:
# top 5 to accumulate based on lowest Gained%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'Gained%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
73,TMPV,3.76,-34.66,128.96,49.60,401810.0,-165295.0,311577.0,600.00,-531.72,40.0,X-MC,17.18,4.0,-0.41,2.02,3.76,XY24,NTT,AUTO
26,GILLETTE,-1.27,-15.40,52.11,28.69,111932.0,-39096.0,214800.0,10891.01,-39.47,47.0,X-SC,6.13,83.0,-0.35,1.40,3.78,X40,ATH,FMCG
6,AWL,-0.32,-41.35,184.77,67.02,386114.0,-147339.0,208970.0,485.00,-74.23,48.0,X-SC,10.53,81.0,-0.38,1.36,3.94,XY24,NTT,FMCG
54,PAGEIND,0.75,-19.04,63.31,32.21,137560.0,-51099.0,217280.0,50690.44,-38.63,31.0,X-SC,6.16,55.0,-0.37,1.41,4.26,X40,ATH,APPARELS
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA


In [21]:
# top 5 to accumulate based on lowest CurrAlloc%
df_tmp = df_portfolio_features[df_portfolio_features['Conviction'].isin(['X-LC','H-LC','X-MC','X-SC'])].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
10,BATAINDIA,0.56,-59.52,234.02,35.20,121882.0,-76588.0,52082.0,2096.00,-19.19,45.0,X-SC,6.52,91.0,-0.63,0.34,8.23,X40,NTT,FOOTWEAR
55,PGHH,1.57,-48.02,152.06,31.02,168762.0,-102536.0,110984.0,17484.40,-60.94,47.0,X-SC,22.55,57.0,-0.61,0.72,4.48,X40,ATH,FMCG
37,HONAUT,0.74,-22.07,93.62,50.88,140687.0,-42563.0,150275.0,58192.80,-38.96,40.0,X-SC,8.97,90.0,-0.30,0.98,14.49,X40N,ATH,ELECTRICAL
2,ACC,3.15,-43.22,207.90,74.81,316490.0,-115896.0,152232.0,3906.00,21.51,64.0,X-SC,11.60,85.0,-0.37,0.99,12.77,XY24,BTT,CEMENT
1,ABBOTINDIA,0.88,-13.80,33.84,15.37,52171.0,-24685.0,154170.0,34390.26,-30.90,46.0,X-MC,0.80,58.0,-0.47,1.00,4.43,X40,ATH,PHARMA


In [22]:
# top 5 to accumulate based on CumlRnk
df_tmp = df_portfolio_features.sort_values(by = 'CumlRnk', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
46,ITC,1.53,-14.25,50.09,28.70,187954.0,-62362.0,375233.0,452.00,-18.77,75.0,X-LC,5.77,1.0,-0.33,2.44,18.10,X40,NTT,FMCG
71,TCS,-1.34,-29.30,79.27,26.75,273547.0,-142993.0,345082.0,4208.34,-43.36,43.0,X-LC,7.29,2.0,-0.52,2.24,19.69,X40,ATH,IT
44,INFY,-1.64,-21.10,80.82,42.66,255613.0,-84581.0,316274.0,1972.00,-42.51,44.0,X-LC,6.62,3.0,-0.33,2.05,13.22,X40,NTT,IT
73,TMPV,3.76,-34.66,128.96,49.60,401810.0,-165295.0,311577.0,600.00,-531.72,40.0,X-MC,17.18,4.0,-0.41,2.02,3.76,XY24,NTT,AUTO
33,HCLTECH,-1.32,-9.20,38.44,25.70,125994.0,-33204.0,327769.0,1837.07,-15.16,41.0,X-LC,9.34,8.0,-0.26,2.13,30.77,X40,ATH,IT


In [23]:
# top 5 for average up
df_tmp = df_portfolio_features[(df_portfolio_features['Dev%_200'] > 5)].sort_values(by = 'CurrAlloc%', ascending=True)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
12,BOSCH-HCIL,1.23,-22.88,29.66,-0.01,29233.0,-29245.0,98560.0,2282.00,772.08,52.0,M-SC,18.82,231.0,-1.00,0.64,67.76,OX40N,BTT,AC
79,VALIANTORG,-1.11,-29.46,146.87,74.15,169507.0,-48192.0,115413.0,838.00,NaN,37.0,NaN,34.99,145.0,-0.28,0.75,71.75,XR,NTT,CHEMICALS
69,TANLA,-3.83,-24.13,183.62,115.17,377189.0,-65343.0,205418.0,1903.93,-17.22,59.0,H-SC,24.54,147.0,-0.17,1.33,81.59,AR,ATH,IT
27,GLAXO,2.77,8.84,24.67,35.69,53508.0,17612.0,216896.0,3380.16,-20.88,58.0,X-MC,6.48,56.0,0.33,1.41,28.91,X40,ATH,PHARMA
25,FINCABLES,1.64,52.68,35.21,106.44,79373.0,77784.0,225428.0,1629.94,-4.52,47.0,M-SC,22.41,220.0,0.98,1.46,70.81,OX40N,ATH,CABLES


In [24]:
# top 5 RSP
df_tmp = df_portfolio_features.sort_values(by = 'RSP', ascending=False)
df_tmp[cols].head()

,Symbol,Today P/L%,Current P/L%,FTT%,OTT%,FTT Amt,Current P/L,Current,FTT,Dev%_PE,RSI_14,Conviction,Spread%,CumlRnk,RRR Ind,CurrAlloc%,Gained%,Criteria,Strategy,Category
69,TANLA,-3.83,-24.13,183.62,115.17,377189.0,-65343.0,205418.0,1903.93,-17.22,59.0,H-SC,24.54,147.0,-0.17,1.33,81.59,AR,ATH,IT
50,M&M,4.66,9.76,7.04,17.48,23264.0,29382.0,330457.0,3762.88,-2.50,73.0,X-LC,4.37,25.0,1.26,2.15,27.78,X40N,ATH,AUTO
81,VOLTAS,8.59,-3.61,50.64,45.20,93590.0,-6927.0,184815.0,1856.00,5.16,72.0,X-MC,16.19,79.0,-0.07,1.20,30.22,XY25,NTT,AC
51,MARUTI,7.30,-0.25,26.31,25.99,89151.0,-860.0,338850.0,17120.06,3.02,72.0,X-LC,8.87,33.0,-0.01,2.20,21.34,XR,ATH,AUTO
61,REPCOHOME,-0.29,-17.97,131.40,89.82,308823.0,-51480.0,235025.0,880.00,NaN,69.0,NaN,8.13,161.0,-0.17,1.53,26.87,XY24,NTT,FINANCE


In [25]:
# Top N allocation
df_tmp = df_portfolio_features.sort_values(by = 'CurrAlloc%', ascending=False)
top_n_values = [10, 25, 50]

sum_df = pd.DataFrame({
    'Top_N': top_n_values,
    'Sum_Alloc%': [df_tmp['CurrAlloc%'].head(n).sum() for n in top_n_values]
})

sum_df

,Top_N,Sum_Alloc%
0,10,22.79
1,25,51.75
2,50,82.53


In [26]:
# market-cap-wise allocation
df_tmp = df_portfolio_features[cols]
df_tmp.groupby(df_tmp['Conviction'].str[-2:])['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
LC    39.07
SC    28.37
MC    21.65
Name: CurrAlloc%, dtype: float64

In [27]:
# criteria-wise allocation
df_portfolio_features.groupby('Criteria')['CurrAlloc%'].sum().sort_values(ascending=False)

Criteria
X40      27.00
XY24     22.79
X40N     19.52
XY25     10.23
XR        9.89
OX40N     6.67
AR        3.20
SR        0.72
Name: CurrAlloc%, dtype: float64

In [28]:
# conviction-wise allocation
df_portfolio_features.groupby('Conviction')['CurrAlloc%'].sum().sort_values(ascending=False)

Conviction
X-LC    37.69
X-MC    20.24
H-SC    14.04
X-SC     8.73
M-SC     5.60
M-LC     1.38
M-MC     1.07
L-MC     0.34
Name: CurrAlloc%, dtype: float64

In [29]:
# sector-wise stats
df_tmp = df_portfolio_features.groupby('Category')[['CurrAlloc%', 'Current', 'Current P/L', 'FTT Amt']].sum().sort_values(by=['Current', 'Current P/L'], ascending=False)
df_tmp['Current P/L%'] = round(df_tmp['Current P/L'] * 100 / df_tmp['Current'], 2)
df_tmp['FTT%'] = round(df_tmp['FTT Amt'] * 100 / df_tmp['Current'], 2)
cols = ['CurrAlloc%', 'Current P/L%', 'FTT%']
df_tmp[cols].sort_values(by=['CurrAlloc%'], ascending=False)

,CurrAlloc%,Current P/L%,FTT%
Category,,,
FMCG,18.66,-22.70,66.95
IT,12.28,-35.56,106.19
FINANCE,10.96,-20.83,72.25
AUTO,9.36,-11.00,51.78
PAINTS,5.59,-23.16,45.28
MISC,5.54,-44.51,102.59
INSURANCE,5.41,0.82,36.46
BANKS,5.30,-9.89,64.99
PHARMA,4.49,1.79,22.87


In [30]:
# money to be made criteria-wise
df_portfolio_features.groupby('Criteria')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Criteria,,
XY24,3986738.0,19
X40,2175856.0,16
XR,1584515.0,13
X40N,1431468.0,12
AR,1013462.0,6
OX40N,760241.0,10
XY25,715927.0,6
SR,303121.0,2


In [31]:
# money to be made conviction-wise
df_portfolio_features.groupby('Conviction')['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

,sum,count
Conviction,,
X-LC,2449048.0,17
H-SC,2282364.0,15
X-MC,1851053.0,14
X-SC,1461637.0,8
M-SC,509369.0,7
M-LC,270773.0,1
M-MC,81174.0,1
L-MC,49998.0,1


In [32]:
# money to be made criteria and conviction-wise
df_portfolio_features.groupby(['Conviction', 'Criteria'])['FTT Amt'].agg(['sum', 'count']).sort_values(by='sum', ascending=False)

sum  count
Conviction Criteria                  
X-LC       X40       1291511.0      8
H-SC       XY24       986133.0      5
X-SC       XY24       702604.0      2
X-MC       XY24       670687.0      3
           X40N       556091.0      5
H-SC       AR         542694.0      2
X-SC       X40        540136.0      4
X-LC       X40N       487356.0      4
H-SC       XR         376968.0      4
X-LC       XY25       354687.0      3
X-MC       X40        344209.0      4
           XY25       280066.0      2
M-LC       XR         270773.0      1
H-SC       OX40N      249855.0      3
X-LC       XY24       226343.0      1
X-SC       X40N       218897.0      2
M-SC       XR         193290.0      2
           OX40N      186839.0      4
           XY24       129240.0      1
H-SC       SR         126714.0      1
X-LC       XR          89151.0      1
M-MC       XY25        81174.0      1
L-MC       XR          49998.0      1

In [33]:
# notebook execution time

end_time = time.time()
execution_time = round(end_time - start_time, 0)
print(f"Notebook execution time: {execution_time} seconds")

Notebook execution time: 20.0 seconds
